# Host, workers, registration, and shutdown: practice

Start gRPC host → start one or several workers → register agents → send judge request → collect replies → stop workers then host.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [5_autogen/4_lab4_autogen_distributed.ipynb](../../../../../5_autogen/4_lab4_autogen_distributed.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Typed messages and agent identities

A runtime delivers a message to an identified handler. The handler owns its work. This dictionary-based simulation exposes type/key routing without starting AutoGen or network listeners.

**Predict:** Will an unregistered type resolve?

In [ ]:
from dataclasses import dataclass
@dataclass
class Message:
    content: str
registry = {("simple_agent", "default"): lambda message: Message("Received: " + message.content)}
recipient = ("simple_agent", "default")  # Agent type and instance key.
response = registry[recipient](Message("Hello"))
print(response.content)
assert response.content == "Received: Hello"

**Check:** No. Registration must precede delivery. The simulation does not verify an AutoGen runtime.

**Change one thing:** Register a second handler and send the response to it. Explain which code routes and which code does the work.

## 2. Host and workers have different responsibilities

A distributed host connects workers; workers register agents and handle requests. A stop sequence must clean up both. This logical model opens no port and starts no process.

**Predict:** If both workers register different agents, who handles the judge request?

In [ ]:
workers = {"worker_a": {"researcher"}, "worker_b": {"judge"}}
def locate(agent_type):
    matches = [worker for worker, agents in workers.items() if agent_type in agents]
    if len(matches) != 1:
        raise LookupError("Need exactly one matching worker in this fixture")
    return matches[0]
print(locate("judge"))
cleanup_order = ["worker_a stopped", "worker_b stopped", "host stopped"]
print(cleanup_order)
assert locate("judge") == "worker_b"

**Check:** worker_b. The real source uses gRPC; this exercise only models the delivery/lifetime concept.

**Change one thing:** Add another agent type to worker_a and locate it. Explain how a host connection failure differs from missing registration.

## 3. Await vs. concurrent work

Calling an async function creates a coroutine. Await executes/waits for it. gather schedules independent coroutines and returns results in input order. Notebook cells support top-level await.

**Predict:** Will results follow completion order or input order?

In [ ]:
import asyncio
async def work(label, delay):
    await asyncio.sleep(delay)  # Yield control while waiting.
    return label
results = await asyncio.gather(work("A", 0.002), work("B", 0))
print(results)
assert results == ["A", "B"]

**Check:** Input order. In a standalone script, wrap work in async def main and call asyncio.run(main()); do not do that inside a running notebook loop.

**Change one thing:** Run the two functions with separate awaits. Explain why a dependent stage cannot start until it has its input.

## 4. A tool schema describes; a dispatcher executes

The model can request a function call. Your application decodes arguments, recognizes the name, executes the function, and returns a matching tool result. No model is needed to practice this boundary.

**Predict:** Does defining the schema call the function?

In [ ]:
import json
records = []
def record_interest(email):
    records.append(email)  # In memory only; does not send email or notifications.
    return {"recorded": True}
schema = {
    "type": "function", "function": {
        "name": "record_interest", "description": "Record a supplied address",
        "parameters": {"type": "object", "properties": {"email": {"type": "string"}},
                       "required": ["email"], "additionalProperties": False},
    },
}
assert records == []
requested = {"id": "call_1", "name": "record_interest", "arguments": '{"email":"learner@example.com"}'}
allowed = {"record_interest": record_interest}  # An explicit registry of permitted functions.
if requested["name"] not in allowed:
    raise ValueError("Unknown tool")
result = allowed[requested["name"]](**json.loads(requested["arguments"]))
tool_message = {"role": "tool", "tool_call_id": requested["id"], "content": json.dumps(result)}
print(tool_message)
assert records == ["learner@example.com"]

**Check:** No. Only the explicit dispatcher call changes records. Keep the assistant tool-call message before the matching tool result in a full conversation.

**Change one thing:** Change the requested name to an unknown function and observe the error. Then add a harmless second tool to the registry.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.